In [ ]:
import os
import sys

ENDWITHS = 'FloresPlus'

NOTEBOOK_DIR = os.getcwd()

if not NOTEBOOK_DIR.endswith(ENDWITHS):
    raise ValueError(f"Not in correct dir, expect end with {ENDWITHS}, but got {NOTEBOOK_DIR} instead")

BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', '..', '..', '..'))
print(BASE_DIR)

sys.path.insert(0, os.path.join(BASE_DIR, 'src'))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [ ]:
from FloresPlusEvaluator import FloresPlusEvaluator
from pipeline.TranslationModels.ElanMtJaEnBatchTranslator import ElanMtJaEnBatchTranslator

In [ ]:
from dotenv import load_dotenv
load_dotenv(os.path.abspath(os.path.join(BASE_DIR, '..', '.env')))

HF_TOKEN = os.getenv('HF_TOKEN')

In [ ]:
OPENMANTRA_ROOT = os.path.join(BASE_DIR, 'data', 'open-mantra-dataset')
print(f"OpenMantra dataset root: {OPENMANTRA_ROOT}")

SAVE_DIR = os.path.join(BASE_DIR, 'output', 'pipeline', 'Evaluators', 'Translators', 'OpenMantra')

In [ ]:
evaluator = FloresPlusEvaluator()

In [ ]:
translation_model = ElanMtJaEnBatchTranslator()

In [ ]:
import re
from typing import List

# Preprocess
def strip_text(xs: List[str]) -> List[str]: 
    return [x.strip() for x in xs]

def normalize_punct(xs: List[str]) -> List[str]: 
    return [x.replace("。", ".").replace("、", ",") for x in xs]

# Postprocess

# def fix_spacing(xs): return [x.replace(" ,", ",") for x in xs]

def truncate_long_predictions(preds: List[str], source_texts: List[str], max_ratio: float = 3.0) -> List[str]:
    """Truncate predictions that are excessively longer than their source."""
    result = []
    for pred, src in zip(preds, source_texts):
        max_len = max(int(len(src) * max_ratio), 50)  # at least 50 chars
        if len(pred) > max_len:
            # Truncate to reasonable length
            result.append(pred[:max_len] + "...")
        else:
            result.append(pred)
    return result

def remove_repetitions(preds: List[str], source_texts: List[str] = None) -> List[str]:
    """Remove obvious repetitive patterns in predictions."""
    import re
    result = []
    for pred in preds:
        # Detect repetitive patterns like "Oh, Oh, Oh," or "a-a-a-a-"
        # Match pattern repeated 3+ times
        cleaned = re.sub(r'(.{2,30}?)\1{3,}', r'\1\1', pred)  # Keep at most 2 repeats
        result.append(cleaned)
    return result

def fix_romanization(preds: List[str], source_texts: List[str] = None) -> List[str]:
    """Fix common romanization artifacts."""
    result = []
    for pred in preds:
        # Remove trailing ellipsis artifacts
        cleaned = re.sub(r'\.{4,}', '...', pred)
        result.append(cleaned)
    return result

# Add preprocess
translation_model.add_preprocess_step("strip", strip_text)
translation_model.add_preprocess_step("normalize_punct", normalize_punct)

# Add postprocess
# translation_model.add_postprocess_step("fix_spacing", fix_spacing)
translation_model.add_postprocess_step("remove_repetitions", remove_repetitions)
translation_model.add_postprocess_step("truncate_long", truncate_long_predictions)
translation_model.add_postprocess_step("fix_romanization", fix_romanization)

translation_model.load_model()

In [ ]:
# Run evaluation
metrics = evaluator.evaluate(translation_model, save_dir=SAVE_DIR)